In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    PrecisionRecallDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    KBinsDiscretizer,
    OneHotEncoder,
    PowerTransformer,
)

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [ ]:
ct = ColumnTransformer(
    [
        (
            "Categorical columns",
            OneHotEncoder(handle_unknown="ignore"),
            make_column_selector(dtype_exclude=int),
        ),
        ("Balance", PowerTransformer(method="yeo-johnson"), ["balance"]),
        (
            "age",
            KBinsDiscretizer(n_bins=5, strategy="quantile", encode="onehot"),
            ["age"],
        ),
    ],
    remainder="passthrough",
)

In [ ]:
dummy_pipe = Pipeline(
    [("preprocessor", ct), ("model", DummyClassifier(strategy="most_frequent"))]
)
logreg_pipe = Pipeline(
    [
        ("preprocessor", ct),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dummy_cv = cross_val_score(
    dummy_pipe, X_train, y_train, cv=cv, scoring="average_precision"
)
log_cv = cross_val_score(
    logreg_pipe, X_train, y_train, cv=cv, scoring="average_precision"
)
print(
    f"mean of dummy classifier: {np.mean(dummy_cv)}, standard deviation: {np.std(dummy_cv)}"
)
print(
    f"mean of logistic regression: {np.mean(log_cv)}, standard deviation: {np.std(log_cv)}"
)

In [ ]:
logreg_pipe.fit(X_train, y_train)
y_proba = logreg_pipe.predict_proba(X_test)[:, 1]
y_pred = logreg_pipe.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
confusion_matrix(y_test, y_pred)

In [ ]:
print(f"test PR-AUC (average precision): {average_precision_score(y_test, y_proba)}")
PrecisionRecallDisplay.from_predictions(y_test, y_proba)
plt.title("PR-AUC curve and AP")
plt.savefig("../reports/figures/PR-AUC.png")